# Learning-to-rank : pyTerrier - OpenNIR

Dans cette partie, on s'intéresse à l'utilisation de modèles neuronaux pour la recherche d'information.
Les modèles neuronaux utilisés ont été rassemblés dans la librairie [OpenNIR](https://opennir.net/).
On explorera également le modèle T5 avec le plugin [monoT5](https://github.com/terrierteam/pyterrier_t5).

In [1]:
!pip install --upgrade python-terrier
!pip install --upgrade git+https://github.com/Georgetown-IR-Lab/OpenNIR
!pip install --upgrade git+https://github.com/terrierteam/pyterrier_t5

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.6 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=4a320b7f1fe9178e73b7fb7e95f8c2d81a2c7cbb56a398a261cebdcb7c9cf0cb
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c


## Initialisation
De façon similaire au TP1, on initialise PyTerrier. Nous allons travailler sur le dataset CORD19. le bloc suivant est une répétition du code en TP1.

In [2]:
import pyterrier as pt
if not pt.started():
    pt.init(tqdm='notebook')
import onir_pt

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


/tmp/ipykernel_380/3869899361.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_380/3869899361.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
The following code will have the same effect:
pt.utils.set_tqdm('notebook')
pt.java.init() # optional, forces java initialisation
  pt.init(tqdm='notebook')


Better speed can be achieved with apex installed from https://www.github.com/nvidia/apex.


In [3]:
import os

dataset = pt.datasets.get_dataset('irds:cord19/trec-covid')
topics = dataset.get_topics(variant='description')
qrels = dataset.get_qrels()

indexer = pt.index.IterDictIndexer('./cord19-index', text_attrs=['title', 'abstract'], fields=True)
indexref = indexer.index(dataset.get_corpus_iter())
index = pt.IndexFactory.of(indexref)



[INFO] [starting] https://ir.nist.gov/covidSubmit/data/topics-rnd5.xml
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/topics-rnd5.xml: s] [18.7kB] [21.6MB/s]
[INFO] [starting] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j0.5-5.txt
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j0.5-5.txt: s] [1.14MB] [3.60MB/s]
[INFO] [starting] building docstore
[INFO] If you have a local copy of https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/80d664e496b8b7e50a39c6f6bb92e0ef
[INFO] [starting] https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv
docs_iter:   0%|                                    | 0/192509 s<?, ?doc/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 0.00/269M s<?, ?B/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/me

cord19/trec-covid documents:   0%|          | 0/192509 s<?, ?it/s]

10:21:14.127 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
10:22:22.622 [ForkJoinPool-1-worker-1] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(In

In [6]:
print(os.path.exists("terrier_index.zip"))

True


In [7]:
#Si le chargement de la collection est trop long à cause du débit ou autres raisons,
# il est possible de récupérer directement l'index fourni par Terrier
import os

if not os.path.exists("terrier_index.zip"):
  !wget http://www.dcs.gla.ac.uk/~craigm/ecir2021-tutorial/terrier_index.zip
  !unzip -j terrier_index.zip -d terrier_index

index_ref = pt.IndexRef.of("./terrier_index/data.properties")
index = pt.IndexFactory.of(index_ref)

## Modèles de ré-ordonancement neuronaux "from scratch"

Les modèles de ré-ordonnancement dans OpenNIR sont constitués de deux éléments :
*  ranker: un modèle d'ordonnancement (e.g., drmm, knrm, pacrr, ...). Cf la liste des [Rankers](https://opennir.net/rankers.html). Il est également possible de rajouter des modèles d'ordonnancement en étendant la classe Ranker.
*  vocab : défnit comment le texte est encodé par le modèle (e.g., wordvec_hash, bert, ...). Cela permet ainsi de tester plusieurs méthodes de représentation. Plus de détails concernant le [vocab](https://opennir.net/vocab.html).

Les modèles de réordonnancement s'appuient sur une première étape d'ordonnancement (souvent BM25), récupèrent ensuite les textes des top documents et appliquent ensuite le modèle neuronal.



In [8]:
knrm = onir_pt.reranker('knrm', 'wordvec_hash', text_field='abstract')


config file not found: config
[2026-04-13 10:23:19,977][WordvecHashVocab][DEBUG] [starting] downloading https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip


[2026-04-13 10:23:33,487][onir.util.download][WARNING] no hash provided for https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip; consider adding expected_md5="3cc8839ac3fa9a6187149b1e73328b2a" to ensure data integrity.
[2026-04-13 10:23:33,491][onir.util.download][DEBUG] downloaded https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip [13.22s] [682M] [54.5MB/s]
[2026-04-13 10:23:33,496][WordvecHashVocab][DEBUG] [finished] downloading https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip [13.52s]
[2026-04-13 10:23:33,497][WordvecHashVocab][DEBUG] [starting] extracting vecs
[2026-04-13 10:23:55,443][WordvecHashVocab][DEBUG] [finished] extracting vecs [21.95s]
[2026-04-13 10:23:55,444][WordvecHashVocab][DEBUG] [starting] loading vecs into memory
[2026-04-13 10:26:09,589][WordvecHashVocab][DEBUG] [finished] loading vecs into memory [02:14]
[2026-04-13 10:26:09,850][WordvecHashVocab][DEBUG] [starting] writ

Une fois le modèle chargé, il est nécessaire de mettre en place la pipeline d'évaluation vue dans le tp1.
Le modèle neuronal n'est pas efficace car il n'a pas été entraîné et utilise des poids aléatoires.

In [9]:
br = pt.BatchRetrieve(index) % 100
pipeline = br >> pt.text.get_text(dataset, 'abstract') >> knrm
pt.Experiment(
    [br, pipeline],
    topics,
    qrels,
    names=['DPH', 'DPH >> KNRM'],
    eval_metrics=["map", "ndcg", 'ndcg_cut.10', 'P.10', 'mrt']
)

/tmp/ipykernel_380/3806215551.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  br = pt.BatchRetrieve(index) % 100


[2026-04-13 10:28:47,997][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:28:48,597][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-04-13 10:28:48,610][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: DPH >> KNRM ((TerrierRetr(DPH) >> RankCutoff(100) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7b22d10fd010> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-04-13 10:28:56,364][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:28:56,365][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-04-13 10:28:59,638][onir_pt][DEBUG] [finished] batches: [3.27s] [1250it] [381.88it/s]


,name,map,P.10,ndcg,ndcg_cut.10,mrt
0,DPH,0.068056,0.658,0.165653,0.609058,3628.122552
1,DPH >> KNRM,0.054808,0.450,0.145487,0.359965,7262.222804


## Entraînement des modèles sur les jeux de données

Pour entraîner les modèles, il est nécessaire de construire la pipeline de modèles/transformations à utliser et ensuite d'appliquer la fonction .fit() à la pipeline.
Le code ci-dessous prend beaucoup de temps. Il est donné à titre indicatif si vous souhaitez l'utiliser sur des serveurs adaptés. L'étape suivante permet de charger directement les poids du modèle pré-entraîné à l'avance et mis à disposition de la communauté.

Dans ce qui suit, on utilise le jeu de données MS MARCO medical pour pré-entraîner le modèle qui sera ensuite appliqué sur CORD19.
Il est également possible de garder seulement le jeu de données CORD19, de le découper en train/val/test et regarder les performances.

In [17]:
# Apprentissage du modèle sur des données médicales (MS MARCO medical)
from sklearn.model_selection import train_test_split
train_ds = pt.datasets.get_dataset('irds:msmarco-passage/train/medical')
train_topics, valid_topics = train_test_split(train_ds.get_topics(), test_size=50, random_state=42)

# Indexation de MS MARCO pour la première étape d'ordonnancement et récupérer les textes (pour le ranker openNIR)
indexer = pt.index.IterDictIndexer(
    './terrier_msmarco-passage',
    text_attrs=['text'],
    meta={'docno': 20} # Specify 'docno' and its maximum byte length here
)
tr_index_ref = indexer.index(train_ds.get_corpus_iter())

pipeline = (pt.BatchRetrieve(tr_index_ref) % 100 # récupère les 100 premiers documents
            >> pt.text.get_text(train_ds, 'text') # récupère le texte de ces documents
            >> pt.apply.generic(lambda df: df.rename(columns={'text': 'abstract'})) # renomme la colonne
            >> knrm) # applique le re-ranker


pipeline.fit(
    train_topics,
    train_ds.get_qrels(),
    valid_topics,
    train_ds.get_qrels())

msmarco-passage/train/medical documents:   0%|          | 0/8841823 s<?, ?it/s]

[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz

https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.0%| 0.00/1.06G s<?, ?B/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.0%| 459k/1.06G s<04:31, 3.90MB/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.2%| 2.21M/1.06G s<01:45, 10.0MB/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.7%| 7.12M/1.06G s<47.53s, 22.1MB/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 1.7%| 18.2M/1.06G s<25.00s, 41.6MB/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/co

KeyboardInterrupt: 

Pour éviter l'étape d'entraînement ici, on utlise une version pré-entraînée.

**Important** : penser à supprimer la mémoire des re-rankers qui vont être mis à jour avec les modèles pré-entrainés.

In [18]:
del knrm # free up the memory before loading a new version of the ranker
knrm = onir_pt.reranker.from_checkpoint('https://macavaney.us/knrm.medmarco.tar.gz', text_field='abstract', expected_md5="d70b1d4f899690dae51161537e69ed5a")

[2026-04-13 10:40:23,161][onir.util.download][DEBUG] downloaded https://macavaney.us/knrm.medmarco.tar.gz s] [1.43k] [7.51MB/s] [md5 hash verified]
[2026-04-13 10:40:23,170][WordvecHashVocab][DEBUG] [starting] reading cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p
[2026-04-13 10:40:32,444][WordvecHashVocab][DEBUG] [finished] reading cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p [9.27s]


Il est ensuire possible de lancer la pipeline d'évaluation avec ce nouveau modèle. Les résultats sont meilleurs !

In [19]:
pipeline = br >> pt.text.get_text(dataset, 'abstract') >> knrm
pt.Experiment(
    [br, pipeline],
    topics,
    qrels,
    names=['DPH', 'DPH >> KNRM'],
    baseline=0,       ## spécifie quelle est le modèle de référence pour calculer les améliorations.
    eval_metrics=["map", "ndcg", 'ndcg_cut.10', 'P.10', 'mrt']
)

[2026-04-13 10:40:38,449][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:40:38,760][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-04-13 10:40:38,774][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: DPH >> KNRM ((TerrierRetr(DPH) >> RankCutoff(100) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7b22aa1644d0> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-04-13 10:40:44,567][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:40:44,567][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-04-13 10:40:47,709][onir_pt][DEBUG] [finished] batches: [3.14s] [1250it] [397.95it/s]


,name,map,P.10,ndcg,ndcg_cut.10,mrt,map +,map -,map p-value,P.10 +,P.10 -,P.10 p-value,ndcg +,ndcg -,ndcg p-value,ndcg_cut.10 +,ndcg_cut.10 -,ndcg_cut.10 p-value
0,DPH,0.068056,0.658,0.165653,0.609058,2069.806677,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DPH >> KNRM,0.065106,0.598,0.160562,0.532655,6726.705509,20.0,30.0,0.09626,12.0,26.0,0.024604,20.0,30.0,0.028326,20.0,30.0,0.005972


**Exercice 1**
Autre pipeline. A vous de deviner ce qu'elle fait !

In [20]:
cutoffs = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
dph = pt.BatchRetrieve(index)
res = pt.Experiment(
    [dph % cutoff >> pt.text.get_text(dataset, 'abstract') >> knrm for cutoff in cutoffs],
    dataset.get_topics('description'),
    dataset.get_qrels(),
    names=[f'c={cutoff}' for cutoff in cutoffs],
    eval_metrics=["map", "recip_rank", "ndcg", "ndcg_cut.10", "mrt"]
)
res

[2026-04-13 10:42:42,638][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:42:42,639][onir_pt][DEBUG] [starting] batches


/tmp/ipykernel_380/2093709227.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  dph = pt.BatchRetrieve(index)


batches: 0it s, ?it/s]

[2026-04-13 10:42:42,653][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]
[2026-04-13 10:42:42,668][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:42:42,668][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

...

batches:   0%|          | 0/1125 s<?, ?it/s]

[2026-04-13 10:43:17,741][onir_pt][DEBUG] [finished] batches: [3.04s] [1125it] [370.15it/s]
[2026-04-13 10:43:20,143][onir_pt][DEBUG] using GPU (deterministic)
[2026-04-13 10:43:20,144][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/1250 s<?, ?it/s]

[2026-04-13 10:43:22,829][onir_pt][DEBUG] [finished] batches: [2.69s] [1250it] [465.54it/s]


,name,map,recip_rank,ndcg,ndcg_cut.10,mrt
0,c=10,0.011837,0.785333,0.049385,0.597078,2581.975493
1,c=20,0.020948,0.787500,0.071907,0.587754,2531.674197
2,c=30,0.029344,0.805667,0.089957,0.598327,3026.065337
3,c=40,0.034927,0.808524,0.103090,0.582940,4115.460821
4,c=50,0.040713,0.825667,0.114788,0.591895,3384.025165
5,c=60,0.046934,0.815667,0.126097,0.589682,3866.670535
6,c=70,0.051910,0.816333,0.136295,0.578130,5064.109338
7,c=80,0.056876,0.804667,0.145015,0.565053,4111.312224
8,c=90,0.061705,0.806333,0.154461,0.569579,4988.100416
9,c=100,0.065106,0.767889,0.160562,0.532655,4856.753739


## Modèle Vanilla BERT

**Exercice 2**

Sur le même principe que le modèle KNRM, analyser les performances du modèle vanilla BERT sans et avec pré-entraînement.
Pour la version du modèle pré-entraîné, on utilisera le checkpoint ['https://macavaney.us/scibert-medmarco.tar.gz']('https://macavaney.us/scibert-medmarco.tar.gz') avec le paramètre expected_md5="854966d0b61543ffffa44cea627ab63b".

Synthétisez toutes les mesures d'évaluation dans un même tableau (bm25, knrm et Vanilla Bert / avec/sans entraînement).

# Autres modèles - hors OpenNIR
PyTerrier a mis a disposition l'implémentation d'autres modèles neuronaux:
- ColBERT : https://github.com/terrierteam/pyterrier_colbert
- T5 : https://github.com/terrierteam/pyterrier_t5
- Doc2Query : https://github.com/terrierteam/pyterrier_doc2query
- DeepCT : https://github.com/terrierteam/pyterrier_deepct
- ANCE : https://github.com/terrierteam/pyterrier_ance

Un exemple de code de code est donnée ci-dessous avec Mono-T5 :

In [ ]:
from pyterrier_t5 import MonoT5ReRanker
monoT5 = MonoT5ReRanker(text_field='abstract')

br = pt.BatchRetrieve(index) % 30
pipeline = (br >> pt.text.get_text(dataset, 'abstract') >> monoT5)
pt.Experiment(
    [br, pipeline],
    dataset.get_topics('description'),
    dataset.get_qrels(),
    names=['DPH', 'DPH >> T5'],
    eval_metrics=["map", "recip_rank", "ndcg", "ndcg_cut.10", "mrt"]
)